In [1]:
import boto3
import json

# Initialize the Bedrock Runtime client
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    region_name='us-east-2'  # Change to your preferred region
)

def get_embedding(text):
    # Model ID for Titan Text Embeddings V2
    model_id = "amazon.titan-embed-text-v2:0"

    # Configure the request body
    body = json.dumps({
        "inputText": text,
        "dimensions": 512,  # Optional: 256, 512, or 1024
        "normalize": True   # Recommended for RAG/similarity search
    })

    # Invoke the model
    response = bedrock_runtime.invoke_model(
        body=body,
        modelId=model_id,
        accept="application/json",
        contentType="application/json"
    )

    # Parse the response
    response_body = json.loads(response.get('body').read())
    return response_body.get('embedding')

# Example usage
text_to_embed = "Amazon Bedrock makes it easy to use foundation models."
vector = get_embedding(text_to_embed)
print(f"Vector length: {len(vector)}")
print(f"First 5 values: {vector[:5]}")

Vector length: 512
First 5 values: [-0.1377226561307907, 0.03562498465180397, -0.045492418110370636, -0.07004348188638687, -0.040860578417778015]


In [2]:
import boto3

bedrock = boto3.client(service_name='bedrock-runtime', region_name='us-east-1')

# Use a specific Model ID or an Inference Profile ID
model_id = "deepseek.v3.2"

messages = [
    {
        "role": "user",
        "content": [{"text": "Explain quantum computing to a five-year-old."}]
    }
]

response = bedrock.converse(
    modelId=model_id,
    messages=messages,
    inferenceConfig={"maxTokens": 512, "temperature": 0.5}
)

print(response['output']['message']['content'][0]['text'])

Imagine you have a **magic coloring box** with special crayons that can be **two colors at once** — like red *and* blue at the same time!

1. **Normal computer** (like your tablet):  
   It solves puzzles one step at a time, like turning one light switch on or off. On… off… on… off… It’s good, but slow for really big puzzles.

2. **Quantum computer**:  
   It’s like having **magic switches that can be on AND off at the same time**.  
   So if you have a big puzzle, it can try **all the paths in the puzzle at once** instead of one by one.

It’s not really magic — it’s a special kind of science using tiny, tiny particles (smaller than atoms!) that can be in two places or two states at once.  
Grown-ups hope to use it to make new medicines, solve space mysteries, or make super-fast computers one day!


In [4]:
import boto3
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize the Boto3 client
# It automatically picks up your AWS credentials from your environment or ~/.aws/credentials
bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1" # Make sure this matches the region where you enabled the model
)

# 2. Instantiate the LangChain Bedrock Model
# We are using Claude 3 Haiku here, but you can swap the model_id
# to any other model you've enabled in Bedrock.
llm = ChatBedrockConverse(
    client=bedrock_client,
    model_id="anthropic.claude-3-haiku-20240307-v1:0",
    temperature=0.7,
    max_tokens=512
)

# 3. Create a Prompt Template
# This allows you to dynamically inject variables into your prompts.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert software engineer who explains complex concepts using simple analogies. Your programming language of choice is {language}."),
    ("human", "Explain {concept} to me in a short paragraph.")
])

# 4. Add an Output Parser
# This converts the complex AI message object returned by the LLM into a simple string.
parser = StrOutputParser()

# 5. Build the Chain
# We use LangChain Expression Language (LCEL) to pipe the components together.
chain = prompt | llm | parser

# 6. Invoke the Chain
print("Calling AWS Bedrock...\n")
response = chain.invoke({
    "language": "Python",
    "concept": "Object-Oriented Programming"
})

print(response)

/Users/user/repos/personal/POC/10k-rag/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Calling AWS Bedrock...

Object-Oriented Programming (OOP) is like building a house. In OOP, each object is like a room in the house, with its own unique features and capabilities. Just like a room has a specific purpose, an object has its own data and functions (called methods) that define what it can do. The different objects in the house (rooms) can interact with each other, just like objects in OOP can interact with one another. The foundation of the house is the class, which is like the blueprint that defines the structure and behavior of all the objects (rooms) within it. OOP allows you to create complex programs by breaking them down into smaller, more manageable pieces (objects), making your code more organized, reusable, and easier to maintain.
